In [ ]:
import sys
import datetime
import zipfile

import geopandas as gpd
import pandas as pd
import r5py
import h3
from shapely.geometry import Polygon, LineString

sys.argv.append("--max-memory")
sys.argv.append("16G")
print("Libraries imported successfully!")


In [ ]:
#Toronto boundary 
toronto_bound = gpd.read_file("data/toronto-boundary-wgs84(1)/citygcs_regional_mun_wgs84.shp")
if toronto_bound.crs != 'EPSG:4326':
    toronto_bound = toronto_bound.to_crs('EPSG:4326')
print(f"Boundary loaded: {toronto_bound.crs}")


In [ ]:
#Hex grid (H3 resolution 9, ~200m) clipped to Toronto boundary 
resolution = 9
min_lng, min_lat, max_lng, max_lat = toronto_bound.total_bounds

geojson_polygon = {
    "type": "Polygon",
    "coordinates": [[
        [min_lng, min_lat], [max_lng, min_lat],
        [max_lng, max_lat], [min_lng, max_lat],
        [min_lng, min_lat]
    ]]
}

hex_ids = h3.geo_to_cells(geojson_polygon, resolution)
print(f"Generated {len(hex_ids):,} hex cells")

hexagons = []
for hex_id in hex_ids:
    boundary = h3.cell_to_boundary(hex_id)
    polygon_coords = [(lng, lat) for lat, lng in boundary]
    hexagons.append({'hex_id': hex_id, 'geometry': Polygon(polygon_coords)})

gdf_hex = gpd.GeoDataFrame(hexagons, crs='EPSG:4326')

toronto_union = toronto_bound.geometry.unary_union
gdf_hex['geometry'] = gdf_hex.geometry.intersection(toronto_union)
gdf_hex = gdf_hex[~gdf_hex.geometry.is_empty].reset_index(drop=True)
gdf_hex['id'] = gdf_hex.index

print(f"Clipped to {len(gdf_hex):,} hex cells within Toronto")


In [ ]:
#Transport network + origins + destinations 
osm_path  = "data/Toronto.osm.pbf"
gtfs_path = "data/Complete GTFS.zip"

print("Building transport network (this takes a few minutes)...")
transport_network = r5py.TransportNetwork(osm_path, [gtfs_path])
print("Network ready!")

#Origins: hex centroids in WGS84 (projected for accuracy then back)
origins_projected = gdf_hex.to_crs('EPSG:32617')
centroids = origins_projected.geometry.centroid.to_crs('EPSG:4326')
origins = gpd.GeoDataFrame({'id': gdf_hex.index, 'geometry': centroids}, crs='EPSG:4326')
print(f"Origins: {len(origins):,} hex centroids")

#Destinations: library locations
libraries_raw = gpd.read_file("data/tpl-branch-general-information - 4326.geojson")
libraries_raw = libraries_raw[libraries_raw.geometry.notna()].reset_index(drop=True)
destinations = gpd.GeoDataFrame({
    'id': libraries_raw.index,
    'geometry': libraries_raw.geometry.representative_point()
}, crs='EPSG:4326')
print(f"Destinations: {len(destinations):,} libraries")


In [ ]:
#Helper: r5py results → GeoJSON 
def results_to_geojson(results, hex_polygons, output_path):
    min_times = (
        results
        .groupby('from_id')['travel_time']
        .min()
        .reset_index()
        .rename(columns={'from_id': 'id', 'travel_time': 'minutes'})
    )
    merged = hex_polygons.merge(min_times, on='id', how='left')
    merged['minutes'] = merged['minutes'].fillna(60)

    def bucket(m):
        if m < 15:   return 'under_15'
        elif m < 30: return '15_to_30'
        else:        return 'over_30'

    merged['time_bucket'] = merged['minutes'].apply(bucket)

    out = merged[['geometry', 'time_bucket', 'minutes']].copy()
    out = out[out.geometry.notna() & ~out.geometry.is_empty]
    if out.crs != 'EPSG:4326':
        out = out.to_crs('EPSG:4326')

    out.to_file(output_path, driver='GeoJSON')
    print(f"Saved {output_path} — {len(out):,} hexagons")
    print(f"  under_15: {(out.time_bucket == 'under_15').sum():,}")
    print(f"  15_to_30: {(out.time_bucket == '15_to_30').sum():,}")
    print(f"  over_30:  {(out.time_bucket == 'over_30').sum():,}")
    return out


In [ ]:
#Walk + Transit, weekday (Tuesday June 8 2026, 10:00) 
print("Running: walk + transit, weekday...")
results_weekday = r5py.TravelTimeMatrix(
    transport_network,
    origins=origins,
    destinations=destinations,
    transport_modes=[r5py.TransportMode.WALK, r5py.TransportMode.TRANSIT],
    departure=datetime.datetime(2026, 6, 8, 10, 0, 0),
)
print(f"Done — {len(results_weekday):,} calculations")
results_to_geojson(results_weekday, gdf_hex, "isochrones_transit_weekday.geojson")


In [ ]:
#Walk + Transit, weekend (Saturday June 13 2026, 10:00) 
print("Running: walk + transit, weekend...")
results_weekend = r5py.TravelTimeMatrix(
    transport_network,
    origins=origins,
    destinations=destinations,
    transport_modes=[r5py.TransportMode.WALK, r5py.TransportMode.TRANSIT],
    departure=datetime.datetime(2026, 6, 13, 10, 0, 0),
)
print(f"Done — {len(results_weekend):,} calculations")
results_to_geojson(results_weekend, gdf_hex, "isochrones_transit_weekend.geojson")


In [ ]:
#Walking only (Tuesday June 8 2026, 10:00) 
print("Running: walk only...")
results_walk = r5py.TravelTimeMatrix(
    transport_network,
    origins=origins,
    destinations=destinations,
    transport_modes=[r5py.TransportMode.WALK],
    departure=datetime.datetime(2026, 6, 8, 10, 0, 0),
)
print(f"Done — {len(results_walk):,} calculations")
results_to_geojson(results_walk, gdf_hex, "isochrones_walk.geojson")


In [ ]:
#Fixing zero-minute hexes (library locations rendered as under_15) 
#Hexes with minutes == 0 are those sitting directly on a library location.
#Reclassifying them as over_30 so they don't render as ghost-white patches.
import json

for fname in ['isochrones_walk.geojson',
              'isochrones_transit_weekday.geojson',
              'isochrones_transit_weekend.geojson']:
    with open(fname) as f:
        gj = json.load(f)
    fixed = 0
    for feature in gj['features']:
        if feature['properties']['minutes'] == 0:
            feature['properties']['time_bucket'] = 'over_30'
            fixed += 1
    with open(fname, 'w') as f:
        json.dump(gj, f)
    print(f"{fname}: fixed {fixed} zero-minute hexes")


In [ ]:
#Exporting libraries.geojson 
libraries_export = libraries_raw[['BranchName', 'Address', 'geometry']].copy()
libraries_export = libraries_export[libraries_export.geometry.notna()].reset_index(drop=True)
libraries_export['geometry'] = libraries_export.geometry.representative_point()
libraries_export.to_file("libraries.geojson", driver='GeoJSON')
print(f"Saved libraries.geojson — {len(libraries_export)} branches")


In [ ]:
#TTC rapid transit lines from GTFS 
# Routes: 1 Yonge-University, 2 Bloor-Danforth, 4 Sheppard,
#         5 Eglinton (LRT), 6 Finch West (LRT)
with zipfile.ZipFile("data/Complete GTFS.zip") as z:
    shapes = pd.read_csv(z.open("shapes.txt"))
    trips  = pd.read_csv(z.open("trips.txt"), low_memory=False)
    routes = pd.read_csv(z.open("routes.txt"))

shape_routes = (
    trips[['shape_id', 'route_id']]
    .drop_duplicates()
    .merge(routes[['route_id', 'route_short_name', 'route_long_name', 'route_type']], on='route_id')
    .drop_duplicates('shape_id')
)

lines = []
for shape_id, group in shapes.sort_values('shape_pt_sequence').groupby('shape_id'):
    coords = list(zip(group['shape_pt_lon'], group['shape_pt_lat']))
    if len(coords) >= 2:
        info = shape_routes[shape_routes['shape_id'] == shape_id]
        lines.append({
            'shape_id':         shape_id,
            'route_short_name': info['route_short_name'].iloc[0] if len(info) else None,
            'route_long_name':  info['route_long_name'].iloc[0]  if len(info) else None,
            'route_type':       info['route_type'].iloc[0]       if len(info) else None,
            'geometry':         LineString(coords)
        })

gdf_routes = gpd.GeoDataFrame(lines, crs='EPSG:4326')

keep_routes = [1.0, 2.0, 4.0, 5.0, 6.0]
main_lines = gdf_routes[gdf_routes['route_short_name'].isin(keep_routes)].copy()
main_lines = main_lines.dissolve(by='route_short_name').reset_index()

print(main_lines[['route_short_name', 'route_long_name', 'route_type']])
print(f"{len(main_lines)} lines total")
main_lines.to_file("ttc_main_lines.geojson", driver='GeoJSON')
print("Saved ttc_main_lines.geojson")